# Tutorial 01 — Preprocess

Load a pre-ICI PBMC scRNA-seq cohort and the Hallmark gene-to-pathway prior.  
Set up the per-cell expression matrix and patient-level label dictionary used by all downstream irAEGIS scripts.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import sys
from pathlib import Path
import numpy as np
import pandas as pd

REPO = Path('..').resolve()
sys.path.insert(0, str(REPO))

from models.iraegis.data_utils import load_cohort_data

In [2]:
# Configuration

COHORT = 'GSE189125_pre_ici'
PRIOR  = str(REPO / 'datasets' / 'resources' / 'pathway_prior.npz')

In [3]:
# Load Cohort and Pathway Prior
(X, obs, gene_names, ct_groups, ct_ids,
 pat_ids, pat_labels, prior_data) = load_cohort_data(
    COHORT, prior_path=PRIOR, prior_genes_only=True,
    split_ct_groups=['T_cells', 'Monocytes', 'Dendritic'])

print(f'cells       : {X.shape[0]:,}')
print(f'genes       : {X.shape[1]}')
print(f'pathways    : {prior_data["mask"].shape[1]}')
print(f'cell types  : {len(ct_groups)} → {ct_groups}')

Loading datasets/processed_h5ad/GSE189125_pre_ici.h5ad ...
  Cohort GSE189125_pre_ici: 29,626 cells
[relabel_by_grade] GSE189125_pre_ici: grade>=3 → 18,881 Yes cells, 10,745 No cells (16 patients)
[cell QC] Dropping 1,280/29,626 cells with <200 genes
  Inferring cell-type groups ...
=== Inferred cell type groups ===
  CD8+ NKT-like cells: 15 patients, 1,983 cells
    CD8+ NKT-like cells                                  1983 cells, 15 patients
  Naive CD4+ T cells: 15 patients, 6,972 cells
    Naive CD4+ T cells                                   6972 cells, 15 patients
  Naive CD8+ T cells: 9 patients, 1,068 cells
    Naive CD8+ T cells                                   1068 cells, 9 patients
  B_cells: 14 patients, 4,674 cells
    Naive B cells                                        4674 cells, 14 patients
  NK_cells: 15 patients, 3,620 cells
    Natural killer  cells                                3620 cells, 15 patients
  Classical Monocytes: 15 patients, 4,773 cells
    Classical Mo

In [4]:
# Patient-Level Label Distribution
unique_pats = sorted(pat_labels.keys())
pat_arr     = np.array([pat_labels[p] for p in unique_pats])
n_yes = int(pat_arr.sum()); n_no = len(pat_arr) - n_yes
print(f'patients: {len(unique_pats)} ({n_yes} irAE Yes / {n_no} irAE No)')

# Per-patient cell counts
pat_id_arr = np.asarray(pat_ids)
counts = pd.Series(pat_id_arr).value_counts().sort_index()
print(f'\ncells per patient: min={counts.min()}  median={int(counts.median())}  '
      f'max={counts.max()}')

patients: 16 (9 irAE Yes / 7 irAE No)

cells per patient: min=205  median=1518  max=4439


In [5]:
# Pathway prior
mask = prior_data['mask']
pw_names = prior_data.get('pathway_names', [f'pathway_{j}' for j in range(mask.shape[1])])
df_pw = pd.DataFrame({
    'pathway': pw_names,
    'n_genes_in_prior': mask.sum(axis=0).astype(int),
})
print(df_pw.head(10).to_string(index=False))
print(f'\nmask shape           = {mask.shape}')
print(f'pathway-gene density = {mask.mean():.4f}')

                         pathway  n_genes_in_prior
           HALLMARK_ADIPOGENESIS               182
    HALLMARK_ALLOGRAFT_REJECTION               172
      HALLMARK_ANDROGEN_RESPONSE                89
           HALLMARK_ANGIOGENESIS                24
        HALLMARK_APICAL_JUNCTION               160
         HALLMARK_APICAL_SURFACE                35
              HALLMARK_APOPTOSIS               148
   HALLMARK_BILE_ACID_METABOLISM                90
HALLMARK_CHOLESTEROL_HOMEOSTASIS                68
            HALLMARK_COAGULATION                99

mask shape           = (5693, 50)
pathway-gene density = 0.0223
